# The explore interface — feature demo

One cell per feature, on the WaterTAP seawater-RO model.

**Start the server** (from the repo root, wait for the first solve to ingest data):

```bash
uv run acquirium server --config deployments/WATERTAP/models/seawater-ro/acquirium.toml
```

**Old vs new in one paragraph** — `aq.explore()` replaces `aq.query()`:
verbs are short (`entity/related/measurement` instead of
`find_entity/find_related/find_data`); every attribute (medium, unit,
process, ...) is one vocabulary shared by filtering (`where`), projection
(`include`), and faceting (`options`/`facets`) instead of a
`filter_by_*` zoo; `Not()` replaces the `exclude=` str/bool footgun;
multi-hop traversal runs as client-side BFS (SPARQL never sees
join-explosive chains) with `nearest=` for closest-match; attribute
predicates are hidden from generic traversal by default; and aliases
default to the class name you typed.

In [1]:
from acquirium import Acquirium                     # constructor now waits for /health
from acquirium.Client.explore import Not, hidden_predicates

acq = Acquirium(server_url="localhost", server_port=8000)

## Build patterns

`entity(cls)` — class as URI or free text (server-resolved); the alias
defaults to what you typed.

In [2]:
acq.explore().entity("pump").metadata()

pump
str
"""wbs:P2"""
"""wbs:intake"""
"""wbs:P1"""


`alias()` names the current node; `uri=` pins an instance (CURIEs work).

In [3]:
acq.explore().entity(uri="wbs:RO").alias("ro").metadata()

ro
str
"""wbs:RO"""


`related(cls)` finds related entities — by default the *nearest* match
within 3 hops of any non-hidden predicate.

In [4]:
acq.explore().entity("pump").related("tank").metadata()

pump,tank
str,str
"""wbs:P1""","""wbs:storage-tank-2"""
"""wbs:intake""","""wbs:ferric-chloride-addition"""
"""wbs:P2""","""wbs:storage-tank-2"""
"""wbs:P1""","""wbs:anti-scalant-addition"""


`via=` restricts traversal to a predicate (repeatable up to
`max_depth`), a list of predicates, or `"any"`; `nearest=False` returns all
matches, `max_depth=0` opts into unbounded.

In [5]:
(acq.explore().entity("System")
 .related("Equipment", via="hasMember", nearest=False)
 .metadata())

System,Equipment
str,str
"""wbs:posttreatment-system""","""wbs:lime-addition"""
"""wbs:seawater-ro-plant""","""wbs:ferric-chloride-addition"""
"""wbs:seawater-ro-plant""","""wbs:lime-addition"""
"""wbs:seawater-ro-plant""","""wbs:co2-addition"""
"""wbs:seawater-ro-plant""","""wbs:backwash-handling"""
…,…
"""wbs:pretreatment-system""","""wbs:cartridge-filtration"""
"""wbs:desalination-system""","""wbs:PXR"""
"""wbs:seawater-ro-plant""","""wbs:anti-scalant-addition"""


`direction="upstream"/"downstream"` walks the s223 piping topology; the
step patterns each direction infers are inspectable constants in
`explore.directions` and can be passed to `via=` for nearest searches.

In [6]:
(acq.explore().entity(uri="wbs:RO")
 .related("pump", direction="upstream")
 .metadata())

0,pump
str,str
"""wbs:RO""","""wbs:P2"""
"""wbs:RO""","""wbs:P1"""


## Measurements

`measurement()` attaches data-bearing points — the source's own plus its
connection points' (`include_connection_points=False` for own only);
keyword attributes filter inline, `Not()` excludes, lists mean OR.

In [7]:
(acq.explore().entity(uri="wbs:RO").alias("ro")
 .measurement(alias="feed", quantity_kind="mass flow rate", medium=Not("brine"))
 .metadata())

ro,feed
str,str
"""wbs:RO""","""wbs:RO-out-flow-mass-tds"""
"""wbs:RO""","""wbs:RO-in-flow-mass-tds"""
"""wbs:RO""","""wbs:RO-in-flow-mass-water"""
"""wbs:RO""","""wbs:RO-out-retentate-flow-mass…"
"""wbs:RO""","""wbs:RO-out-flow-mass-water"""


On an empty query, `measurement()` is the root form: every registered
stream in the plant (`frm="*"` / `frm=["a", "b"]` attach per-entity).

In [8]:
acq.explore().measurement(quantity_kind="pressure").metadata()

data
str
"""wbs:RO-out-retentate-pressure"""
"""wbs:RO-in-pressure"""
"""wbs:P1-out-pressure"""
"""wbs:RO-out-pressure"""
"""wbs:PXR-brine-out-pressure"""
"""wbs:conn-cartridge-filtration-…"


`measurement(direction=..., nearest=True)` finds the closest up/downstream
measurement matching the filters.

In [9]:
(acq.explore().entity(uri="wbs:P1").alias("p1")
 .measurement(direction="downstream", nearest=True, quantity_kind="pressure")
 .metadata())

p1,p1_downstream_data
str,str
"""wbs:P1""","""wbs:P1-out-pressure"""


## Filter, project, shape

`where()` filters any node by alias (`target=`), same attribute vocabulary
everywhere.

In [11]:
## needs a server running this branch (process is its own resolver kind)
(acq.explore().entity("Equipment").where(process="reverse osmosis")
 .metadata())

Equipment
str
"""wbs:RO"""


`include()` adds `alias.attr` columns (placed right after their node's
column); `required=True` drops rows lacking the attribute.

In [12]:
(acq.explore().entity(uri="wbs:RO").measurement(alias="m")
 .include("quantity_kind", "unit")
 .metadata())

0,m,m.quantity_kind,m.unit
str,str,str,str
"""wbs:RO""","""wbs:RO-in-pressure""","""qudtqk:Pressure""","""None"""
"""wbs:RO""","""wbs:RO-in-temperature""","""qudtqk:Temperature""","""None"""
"""wbs:RO""","""wbs:RO-in-flow-mass-water""","""qudtqk:MassFlowRate""","""unit:KiloGM-PER-SEC"""
"""wbs:RO""","""wbs:RO-out-retentate-pressure""","""qudtqk:Pressure""","""None"""
"""wbs:RO""","""wbs:RO-out-pressure""","""qudtqk:Pressure""","""None"""
…,…,…,…
"""wbs:RO""","""wbs:RO-out-flow-mass-tds""","""qudtqk:MassFlowRate""","""unit:KiloGM-PER-SEC"""
"""wbs:RO""","""wbs:RO-in-flow-mass-tds""","""qudtqk:MassFlowRate""","""unit:KiloGM-PER-SEC"""
"""wbs:RO""","""wbs:RO-out-retentate-flow-mass…","""qudtqk:MassFlowRate""","""unit:KiloGM-PER-SEC"""


`drop()` keeps a node in the pattern but out of the output (rows
deduplicate accordingly); `refocus()` moves the pointer back to an alias.

In [13]:
(acq.explore().entity(uri="wbs:pretreatment-system").drop()
 .related("equipment").measurement(alias="sensor")
 .metadata())

equipment,sensor
str,str
"""wbs:cartridge-filtration""","""wbs:cartridge-filtration-out-t…"
"""wbs:intake""","""wbs:intake-in-toc-concentratio…"
"""wbs:intake""","""wbs:intake-in-tss-concentratio…"
"""wbs:intake""","""wbs:intake-in-tds-concentratio…"
"""wbs:intake""","""wbs:intake-in-flow-rate"""


## Faceted exploration

`options(attr)` — the distinct values of one attribute across the current
matches, counted client-side.

In [14]:
acq.explore().measurement().options("quantity_kind")

quantity_kind,count
str,i64
"""qudtqk:MassFlowRate""",10
"""qudtqk:Pressure""",6
"""qudtqk:MassConcentration""",5
"""qudtqk:Efficiency""",3
"""qudtqk:Power""",2
"""qudtqk:Temperature""",2
"""qudtqk:VolumeFlowRate""",2
"""qudtqk:Area""",1
"""qudtqk:Density""",1


`facets()` — every applicable attribute at once, falling back to
model-wide then ontology vocabulary when the pattern is empty.

In [16]:
acq.explore().entity(uri="wbs:RO").alias("RO").measurement().facets()

FacetSummary('RO_data')
  type [matched]: s223:QuantifiableObservableProperty (11), ns1:VirtualPoint (11)
  medium [matched]: s223:Fluid-Water (3), nawi:Water-Seawater (2), nawi:Water-Brine (1)
  substance [matched]: nawi:Constituent-Salt (3)
  quantity_kind [matched]: qudtqk:MassFlowRate (6), qudtqk:Pressure (3), qudtqk:Area (1), qudtqk:Temperature (1)
  unit [matched]: unit:KiloGM-PER-SEC (6), unit:M2 (1)
  enumeration_kind: (no values)
  data_source: (no values)

## Data

`data()` returns the lazy DataObject; `dataframe(shape="wide")` puts
`time` first and value columns in alphabetical order; `convert_to` resolves
the target unit *jointly with the source* so only convertible matches win.

In [17]:
d = (acq.explore().measurement(alias="tds", quantity_kind="mass concentration")
     .data(cast_value="float"))
d.dataframe(shape="wide").tail(3)

time,tds__wbs:cartridge-filtration-out-toc-concentration,tds__wbs:intake-in-tds-concentration,tds__wbs:intake-in-toc-concentration,tds__wbs:PXR-brine-out-tds-concentration,tds__wbs:storage-tank-3-out-tds-concentration
"datetime[μs, UTC]",f64,f64,f64,f64,f64
2026-08-05 16:45:08.195866 UTC,0.00193,36.44248,0.003016,59.596495,0.222713
2026-08-05 16:45:23.361712 UTC,0.002303,36.353926,0.003598,59.822193,0.2234
2026-08-05 16:45:38.752315 UTC,0.002767,37.171082,0.004322,61.403682,0.23654


In [19]:
d.convert_to("mg/L").dataframe(shape="wide").tail(3)

time,tds__wbs:cartridge-filtration-out-toc-concentration,tds__wbs:intake-in-tds-concentration,tds__wbs:intake-in-toc-concentration,tds__wbs:PXR-brine-out-tds-concentration,tds__wbs:storage-tank-3-out-tds-concentration
"datetime[μs, UTC]",f64,f64,f64,f64,f64
2026-08-05 16:45:08.195866 UTC,1.930431,36442.480043,3.01564,59596.495366,222.713041
2026-08-05 16:45:23.361712 UTC,2.30304,36353.925711,3.597654,59822.192831,223.399907
2026-08-05 16:45:38.752315 UTC,2.76654,37171.081672,4.321688,61403.682303,236.540364


## Guard rails

Attribute predicates, `subClassOf`, `hasProperty`, and `s223:cnx` are hidden
from `via="any"` traversal by default (`hide()`/`unhide()` adjust); every
attribute-taking method documents the attribute table in its docstring
(`help(q.where)`); and inspect any query with `to_sparql()` before running.

In [20]:
sorted(hidden_predicates())[:5]

['http://data.ashrae.org/standard223#cnx',
 'http://data.ashrae.org/standard223#hasConnectionPoint',
 'http://data.ashrae.org/standard223#hasMedium',
 'http://data.ashrae.org/standard223#hasProperty',
 'http://data.ashrae.org/standard223#ofMedium']

In [21]:
print(acq.explore().entity("pump").measurement(alias="m").to_sparql())

SELECT DISTINCT ?v0 ?v1 ?ext1 ?unit1 ?extunit1
WHERE {
  ?v0 <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> ?v0_typ .
  { SELECT DISTINCT ?v0_typ WHERE { ?v0_typ <http://www.w3.org/2000/01/rdf-schema#subClassOf>* <http://data.ashrae.org/standard223#Pump> . } }
  { ?v0 ?p_e0_1 ?v1 . } UNION { ?v0 <http://data.ashrae.org/standard223#hasConnectionPoint> ?cp_e0_k1 . ?cp_e0_k1 ?p_e0_1 ?v1 . }
  ?v1 <https://brickschema.org/schema/Brick/ref#hasExternalReference> ?ext1 .
  OPTIONAL { ?v1 <http://qudt.org/schema/qudt/hasUnit> ?unit1 . }
  OPTIONAL { ?ext1 <http://qudt.org/schema/qudt/hasUnit> ?extunit1 . }
}
